In [ ]:
# =============================================================================
# REGRESSION INFERENCE NOTEBOOK
# Input: CSV with SMILES column
# Output: CSV with HOMO and LUMO predictions
# =============================================================================

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors

print("✓ All imports successful")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

INPUT_PATH = '../../../data/003_kaggle_competition_data/25261_regression_sample_submission.csv'
MODEL_PATH = '../../../models/regression_model.joblib'
OUTPUT_PATH = '../../../data/003_kaggle_competition_data/submission_regression.csv'

print(f"Input:  {INPUT_PATH}")
print(f"Model:  {MODEL_PATH}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
print("="*60)
print("LOADING MODEL")
print("="*60)

model_package = joblib.load(MODEL_PATH)

var_sel = model_package['var_sel']
imp = model_package['imp']
model_homo = model_package['model_homo']
sel_homo = model_package['sel_homo']
model_lumo = model_package['model_lumo']
sel_lumo = model_package['sel_lumo']
homo_min = model_package['homo_min']
homo_max = model_package['homo_max']
lumo_min = model_package['lumo_min']
lumo_max = model_package['lumo_max']

print("✓ Model loaded")
print(f"HOMO CV MAE: {model_package['mae_homo']:.4f}")
print(f"LUMO CV MAE: {model_package['mae_lumo']:.4f}")

In [ ]:
print("="*60)
print("LOADING INPUT DATA")
print("="*60)

df_input = pd.read_csv(INPUT_PATH)
smiles_list = df_input['smiles'].tolist()

print(f"✓ Loaded {len(smiles_list)} molecules")

In [ ]:
print("="*60)
print("COMPUTING DESCRIPTORS")
print("="*60)

descriptor_names = [x[0] for x in Descriptors._descList]

def compute_descriptors(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    return [getattr(Descriptors, name)(mol) for name in descriptor_names]

X_input = pd.DataFrame(
    [compute_descriptors(s) for s in smiles_list],
    columns=descriptor_names
)

print(f"✓ Descriptors computed: {X_input.shape}")

In [ ]:
print("="*60)
print("PREPROCESSING")
print("="*60)

X_clean = var_sel.transform(X_input)
X_imp = imp.transform(X_clean)

print(f"✓ Preprocessed: {X_imp.shape}")

In [ ]:
print("="*60)
print("GENERATING PREDICTIONS")
print("="*60)

# HOMO
pred_homo = model_homo.predict(sel_homo.transform(X_imp))
pred_homo = np.clip(pred_homo, homo_min, homo_max)

# LUMO
pred_lumo = model_lumo.predict(sel_lumo.transform(X_imp))
pred_lumo = np.clip(pred_lumo, lumo_min, lumo_max)

print(f"HOMO: mean={pred_homo.mean():.4f}, std={pred_homo.std():.4f}")
print(f"LUMO: mean={pred_lumo.mean():.4f}, std={pred_lumo.std():.4f}")

In [ ]:
print("="*60)
print("SAVING SUBMISSION")
print("="*60)

submission = pd.DataFrame({
    'smiles': smiles_list,
    'homo': pred_homo,
    'lumo': pred_lumo
})

submission.to_csv(OUTPUT_PATH, index=False)

print(f"✓ Saved to: {OUTPUT_PATH}")
print(f"\nFirst 10 predictions:")
print(submission.head(10).to_string(index=False))